# 03buildplaces: place names and the hover card

PF7, "Kita: Malaysia in 3.2 million dots". Two small files that help the reader find their way:

1. `docs/data/towns.json`: town labels drawn on top of the dots. The list is `data/places/towns.csv` (hand-made; edit it freely). `tier` 1 shows always, 2 from zoom 6.5, 3 from zoom 8.5.
2. `docs/data/areas.geojson`: the 613 small areas (600 state constituencies, DUN, plus the 13 parliamentary seats of the Federal Territories) with their Census 2020 numbers, for the card that follows the mouse.

The card shows census shares for the whole area, not counts of the modelled dots. DUN figures give Bumiputera as one group, so the card does too.

## 0. Settings

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import geopandas as gpd

ROOT = Path.cwd()                                   # <- run from the PF7 folder
RAW = ROOT / "data" / "raw"
TOWNS_CSV = ROOT / "data" / "places" / "towns.csv"  # <- the town list
WEB = ROOT / "docs" / "data"
WEB.mkdir(parents=True, exist_ok=True)

SIMPLIFY_DEG = 0.0008                               # <- shape simplification (degrees; 0.0008 is about 90 m)
DECIMALS = 4                                        # <- decimals kept in coordinates (4 is about 11 m)
TOWN_MAX_OFF_M = 1000                               # <- a town may sit at most this far outside its state's shape (coastal towns)

## 1. Towns

Each town is checked against the state shapes: it must fall inside its own state, or within 1 km of it (towns on the coast can sit just outside a simplified coastline).

In [2]:
towns = pd.read_csv(TOWNS_CSV)
states = gpd.read_file(RAW / "administrative_1_state.geojson")
pts = gpd.GeoDataFrame(towns, geometry=gpd.points_from_xy(towns["lon"], towns["lat"]), crs=4326).to_crs(3857)
near = gpd.sjoin_nearest(pts, states[["state", "geometry"]].rename(columns={"state": "shape_state"}).to_crs(3857), distance_col="off_m")
near = near[~near.index.duplicated(keep="first")]
problems = near[(near["state"] != near["shape_state"]) | (near["off_m"] > TOWN_MAX_OFF_M)]
print(f"{len(towns)} towns: tier 1 {(towns['tier'] == 1).sum()}, tier 2 {(towns['tier'] == 2).sum()}, tier 3 {(towns['tier'] == 3).sum()}")
print("towns outside their state shape:", near.loc[near["off_m"] > 0, ["name", "off_m"]].round(0).to_dict("records"))
print("problems:", "none" if problems.empty else problems[["name", "state", "shape_state", "off_m"]].to_string())

64 towns: tier 1 14, tier 2 19, tier 3 31
towns outside their state shape: [{'name': 'Tawau', 'off_m': 122.0}, {'name': 'Butterworth', 'off_m': 15.0}, {'name': 'Semporna', 'off_m': 353.0}]
problems: none


In [3]:
town_list = [{"name": r.name, "lon": round(float(r.lon), 4), "lat": round(float(r.lat), 4), "tier": int(r.tier)} for r in towns.itertuples()]
(WEB / "towns.json").write_text(json.dumps(town_list))
print(f"towns.json {(WEB / 'towns.json').stat().st_size / 1e3:.1f} kB")

towns.json 4.2 kB


## 2. Small areas with their census numbers

Ethnicity in the DUN file is given as percent of citizens; here it becomes a share of everyone in the area, with non-citizens as their own share, so the shares add up to 100%.

In [4]:
dun = pd.read_csv(RAW / "census_dun.csv")
par = pd.read_csv(RAW / "census_parlimen.csv")
ft = par[par["state"].str.startswith("W.P.")].copy()
dun["sa"] = dun["code_state_dun"]
dun["name"] = dun["dun"]
dun["seat"] = dun["parlimen"]
ft["sa"] = ft["code_state"].astype(str) + "_" + ft["code_parlimen"]
ft["name"] = ft["parlimen"]
ft["seat"] = ""
cols = ["sa", "name", "seat", "state", "population_total", "nationality_citizen", "nationality_non_citizen", "ethnicity_proportion_bumi", "ethnicity_proportion_chinese", "ethnicity_proportion_indian", "ethnicity_proportion_other", "household_total", "area_km2"]
areas = pd.concat([dun[cols], ft[cols]], ignore_index=True)
eth = areas[["ethnicity_proportion_bumi", "ethnicity_proportion_chinese", "ethnicity_proportion_indian", "ethnicity_proportion_other"]].to_numpy() / 100
eth = eth / eth.sum(axis=1, keepdims=True)
cit_share = (areas["nationality_citizen"] / areas["population_total"]).to_numpy()
shares = np.column_stack([eth * cit_share[:, None], 1 - cit_share])
for i, key in enumerate(["bumi", "chinese", "indian", "other", "noncit"]):
    areas["s_" + key] = np.round(shares[:, i] * 100, 1)
print(len(areas), "areas")
print(areas[["name", "state", "population_total", "s_bumi", "s_chinese", "s_indian", "s_other", "s_noncit"]].head(3).to_string(index=False))

613 areas
            name  state  population_total  s_bumi  s_chinese  s_indian  s_other  s_noncit
N.01 Titi Tinggi Perlis             18696    78.4       15.1       2.8      1.0       2.7
     N.02 Beseri Perlis             18129    87.8        4.8       0.6      0.9       5.9
    N.03 Chuping Perlis             24239    81.8        3.7       2.8      8.2       3.5


In [5]:
g_dun = gpd.read_file(RAW / "electoral_1_dun.geojson")
g_par = gpd.read_file(RAW / "electoral_0_parlimen.geojson")
g_dun["sa"] = g_dun["code_state_dun"]
g_ft = g_par[g_par["state"].str.startswith("W.P.")].copy()
g_ft["sa"] = g_ft["code_state"].astype(str) + "_" + g_ft["code_parlimen"]
shapes = gpd.GeoDataFrame(pd.concat([g_dun[["sa", "geometry"]], g_ft[["sa", "geometry"]]], ignore_index=True), geometry="geometry", crs=g_dun.crs)
shapes["geometry"] = shapes.geometry.simplify(SIMPLIFY_DEG, preserve_topology=True)
keep = ["sa", "name", "seat", "state", "population_total", "household_total", "area_km2", "s_bumi", "s_chinese", "s_indian", "s_other", "s_noncit"]
web = shapes.merge(areas[keep], on="sa", how="left")
web = web.rename(columns={"population_total": "pop", "household_total": "hh", "area_km2": "km2"})
print("areas without numbers:", web["pop"].isna().sum())
web.to_file(WEB / "areas.geojson", driver="GeoJSON", COORDINATE_PRECISION=DECIMALS)
print(f"areas.geojson {(WEB / 'areas.geojson').stat().st_size / 1e6:.2f} MB")

areas without numbers: 0
areas.geojson 2.10 MB


## 3. Check: totals

In [6]:
print(f"people in areas.geojson: {int(web['pop'].sum()):,} (census 32,447,385)")
print(web[["s_bumi", "s_chinese", "s_indian", "s_other", "s_noncit"]].sum(axis=1).describe().round(1).to_string())

people in areas.geojson: 32,447,385 (census 32,447,385)
count    613.0
mean     100.0
std        0.1
min       99.8
25%      100.0
50%      100.0
75%      100.0
max      100.1


___

**END**